# Thí nghiệm 5 + 7 — Prompt sensitivity và CoT ablation trên Llama-3-8B

Chạy **cùng một model, cùng một tập 181 ca, cùng một bộ chấm** với ba prompt khác nhau,
để đo xem Finding C phụ thuộc bao nhiêu vào cách viết prompt.

- **V1 — Vietnamese baseline**: prompt gốc của TN3, chạy lại để kiểm tính tất định.
- **V2 — English medical**: cùng nội dung, viết bằng tiếng Anh (câu hỏi vẫn tiếng Việt).
- **V3 — Chain-of-Thought**: bắt model suy luận bốn bước trước khi trả JSON. Đây chính là TN7.

| Range Hit@1 giữa 3 variants | Ngụ ý | Ảnh hưởng bài |
|---|---|---|
| ≤ 2 pp | Finding C robust across prompts | Thêm 1 câu, claim MẠNH HƠN |
| 2–5 pp | Prompt-dependent nhưng cùng hướng | Giữ Finding C kèm caveat |
| > 5 pp, hoặc có variant mất significance | Prompt-fragile | Reframe: *"under baseline prompt"* |

---

## Mốc TN3 để đối chiếu (đã tính lại từ `tn3_perquery_llama3_8b.csv` ngày 17/08/2026)

| Chỉ số | Giá trị |
|---|---|
| Hit@1 tổng (n=181) | **12,71%** (23/181) |
| KTC 95% Wilson | [8,62; 18,35] |
| Hit@1 reachable (n=118) | 17,80% |
| Hit@1 out-of-reach (n=63) | 3,17% |
| MRR | 0,1358 |
| McNemar vs MedKG-HRR, **n=181** | b=22, c=5, p=0,0015 |
| McNemar vs MedKG-HRR, **n=118 reachable** | b=20, c=5, **p=0,0041** |

Con số p=0,0041 trong phiếu là bản **n=118**; tệp `tn3_mcnemar_llama3_8b.csv` nộp trước
đó ghi p=0,0015 vì tính trên cả 181 ca. Hai con số đều đúng, khác tập con — notebook này
in cả hai để không lẫn nữa.

---

**Runtime:** `Runtime → Change runtime type → T4 GPU`. Cấu hình giữ **nguyên xi TN3**:
`NousResearch/Meta-Llama-3-8B-Instruct`, 4-bit NF4, greedy decoding. Đổi bất cứ thứ gì
trong đó là mất quyền so V1 với 12,71%.

**Thời gian:** V1 và V2 mỗi variant ~15–25 phút; V3 dài hơn (~2× token) nên ~30–50 phút.
Tổng khoảng 1–2 giờ. Notebook lưu CSV **sau mỗi variant**, nên Colab có ngắt giữa chừng
thì chỉ mất variant đang chạy.

## 1. Kiểm GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), \
    "KHONG CO GPU. Vao Runtime -> Change runtime type -> T4 GPU roi chay lai."
p = torch.cuda.get_device_properties(0)
print(f"\nGPU : {p.name}")
print(f"VRAM: {p.total_memory/1e9:.2f} GB")

## 2. Cài thư viện

In [ ]:
%pip install -q -U "transformers>=4.43" accelerate bitsandbytes huggingface_hub python-docx

import transformers, torch
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)

## 3. Nạp dữ liệu đầu vào

Chọn **`tn5_input.zip`** (cùng nội dung `tn3_input.zip`). Nếu chép thêm
`tn3_perquery_llama3_8b.csv` vào gói thì Phần 10 kiểm tái lập V1 **ở mức từng ca**,
chặt hơn nhiều so với chỉ so con số tổng.

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs('/content/tn5_data', exist_ok=True)
up = files.upload()
for ten in up:
    if ten.endswith('.zip'):
        with zipfile.ZipFile(ten) as z:
            z.extractall('/content/tn5_data')
    else:
        os.replace(ten, f'/content/tn5_data/{ten}')

print()
for f in sorted(os.listdir('/content/tn5_data')):
    print(' ', f)

## 4. Đọc và kiểm dữ liệu — 181 / 13.081 / 118–63

In [ ]:
MODEL_ID = 'NousResearch/Meta-Llama-3-8B-Instruct'   # y nguyen TN3
TAG      = 'llama3_8b'

import os
import pandas as pd

DATA = '/content/tn5_data'
OUT  = '/content/tn5_ket_qua'
os.makedirs(OUT, exist_ok=True)

df_test = pd.read_csv(f'{DATA}/independent_scored_perquery.csv')
df_icd  = pd.read_csv(f'{DATA}/ICD10_cleaned.csv')

valid_codes_full  = set(df_icd['Mã ICD'].astype(str).str.strip().str.upper())
valid_codes_3char = set(c[:3] for c in valid_codes_full)

n_reach = int((df_test['voi_toi_duoc'] == True).sum())
n_outr  = int((df_test['voi_toi_duoc'] == False).sum())

print(f"So ca test          : {len(df_test):>6}   (ky vong 181)")
print(f"So ma ICD-10 day du : {len(valid_codes_full):>6}   (ky vong 13081)")
print(f"Reachable / out     : {n_reach} / {n_outr}   (ky vong 118 / 63)")

assert len(df_test) == 181,            "SAI so ca test"
assert len(valid_codes_full) == 13081, "SAI so ma ICD"
assert (n_reach, n_outr) == (118, 63), "SAI ty le reachable"

CO_TN3_PERQUERY = os.path.exists(f'{DATA}/tn3_perquery_llama3_8b.csv')
print("\nCo tn3_perquery_llama3_8b.csv =>",
      "kiem tai lap V1 o muc TUNG CA." if CO_TN3_PERQUERY else
      "chi kiem tai lap V1 o muc CON SO TONG.")

## 5. Ba prompt variant

**Cảnh báo cú pháp — chỗ này phiếu in sai và sẽ nổ ngay dòng đầu.**
Ba template trong phiếu chứa ví dụ JSON viết bằng ngoặc nhọn đơn
`{"top10_icd": [...]}`. Vì hàm sinh dùng `str.format()`, ngoặc nhọn đơn bị hiểu là
tên trường thay thế → `KeyError: '"top10_icd"'` ở ca đầu tiên. Phải viết **đôi**
`{{"top10_icd": …}}`, đúng như TN3 đã làm. Ba template dưới đã sửa; cell cuối phần này
`format()` thử cả ba để chứng minh không còn lỗi.

**V1 là bản sao nguyên văn TN3**, đã đối chiếu bằng SHA-256. Không sửa một ký tự.

In [ ]:
PROMPT_V1 = """Bạn là bác sĩ trợ lý chuyên chẩn đoán bằng mã ICD-10.

NHIỆM VỤ: Đọc mô tả triệu chứng bệnh nhân, đưa ra danh sách TOP-10 mã ICD-10
có khả năng nhất, sắp xếp theo xác suất giảm dần.

QUY TẮC:
1. Mỗi mã ICD-10 phải là mã HỢP LỆ (định dạng: 1 chữ cái + 2-3 chữ số +
   tùy chọn "." + 1-2 chữ số. Ví dụ: A00, B07.9, K21.9).
2. Được phép dùng BẤT KỲ mã nào trong toàn bộ catalogue ICD-10 của
   Bộ Y tế Việt Nam (Quyết định 4469/QĐ-BYT 2020), gồm khoảng 13.081 mã.
3. Đưa mã 3 ký tự (phân nhóm) nếu không đủ tự tin về ký tự thứ 4.
4. KHÔNG giải thích. Chỉ trả về JSON theo mẫu.

VÍ DỤ:
Mô tả: "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ, đã 3 ngày."
Kết quả JSON: {{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Mô tả: "{query_text}"
Kết quả JSON:"""

In [ ]:
PROMPT_V2 = """You are a medical AI assistant specializing in ICD-10 coding.

TASK: Given a patient description (which may be in Vietnamese), output the TOP-10
most likely ICD-10 codes from the full Ministry of Health catalogue
(approximately 13,081 codes), sorted by probability descending.

RULES:
1. Each code must be a valid ICD-10 code (format: 1 letter + 2-3 digits
   + optional "." + 1-2 digits, e.g., A00, B07.9, K21.9).
2. Any code from the full ICD-10 catalogue is allowed. Do NOT restrict
   to any subset.
3. Return 3-character codes if unsure about the 4th character.
4. Output JSON only. No explanation.

EXAMPLE:
Description (Vietnamese): "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ."
Output: {{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Description: "{query_text}"
Output:"""

In [ ]:
PROMPT_V3 = """Bạn là bác sĩ trợ lý chuyên chẩn đoán ICD-10.

Đọc mô tả bệnh nhân và thực hiện suy luận từng bước:

Bước 1: Liệt kê 3-5 triệu chứng chính từ mô tả.
Bước 2: Nêu 2-3 chẩn đoán phân biệt có thể (differential diagnosis).
Bước 3: Cho mỗi chẩn đoán, nêu mã ICD-10 tương ứng.
Bước 4: Mở rộng thành TOP-10 mã ICD-10 sắp xếp theo xác suất giảm dần.

Cuối cùng, TRẢ VỀ JSON: {{"top10_icd": [...]}}

VÍ DỤ:
Mô tả: "Em bị đau bụng nhiều, buồn nôn, sốt nhẹ, đã 3 ngày."
Bước 1: triệu chứng chính: đau bụng, buồn nôn, sốt nhẹ, kéo dài 3 ngày
Bước 2: viêm ruột thừa (K35), viêm dạ dày ruột (K52), nhiễm khuẩn tiêu hóa (A09)
Bước 3: K35.9, K52.9, A09
Bước 4: Top-10 sắp xếp theo probability:
{{"top10_icd": ["K35.9", "K52.9", "K37", "K59.0", "R10.4", "K85.9", "R11", "A09", "K80", "R50.9"]}}

---
Mô tả: "{query_text}"
"""

### `max_new_tokens` cho từng variant

| Variant | Giá trị | Lý do |
|---|---|---|
| V1 | **200** | Đúng giá trị TN3 đã chạy. Phiếu ghi 250; đổi lên 250 thì với ca nào chạm trần ở TN3, output sẽ dài thêm và **V1 không còn tái lập được 12,71%** nữa. Giữ 200. |
| V2 | 250 | Theo phiếu. Không có mốc tái lập nào để giữ. |
| V3 | 600 | Theo phiếu. CoT sinh ~2× token; thiếu trần thì JSON bị cắt. |

In [ ]:
PROMPTS = {
    'V1_baseline': PROMPT_V1,
    'V2_english' : PROMPT_V2,
    'V3_cot'     : PROMPT_V3,
}
MAX_TOKENS = {'V1_baseline': 200, 'V2_english': 250, 'V3_cot': 600}

# Chung minh ca ba template format() duoc - phieu goc se nem KeyError o day
for ten, tpl in PROMPTS.items():
    try:
        _ = tpl.format(query_text='<cau hoi>')
        print(f"  {ten:<12} format() OK  ({len(tpl)} ky tu, max_new_tokens={MAX_TOKENS[ten]})")
    except Exception as e:
        raise AssertionError(f"{ten} loi format: {type(e).__name__}: {e}")

import hashlib
SHA_V1 = hashlib.sha256(PROMPT_V1.encode('utf-8')).hexdigest()
print("\nSHA-256 prompt V1:", SHA_V1)
print("(phai trung SHA prompt cua notebook TN3 va TN4)")

## 6. Bộ chấm — nguyên văn TN3, dùng chung cho cả ba variant

In [ ]:
import json, re

def chuan_hoa_ma_icd(raw_code):
    if not raw_code:
        return None
    code = str(raw_code).strip().upper().rstrip('.')
    m = re.match(r'^([A-Z]\d{2}(?:\d)?(?:\.\d{1,2})?)$', code)
    return m.group(1) if m else None


def extract_json_safe(raw):
    m = re.search(r'\{[^{}]*"top10_icd"\s*:\s*\[[^\]]*\][^{}]*\}', raw)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {'top10_icd': re.findall(r'\b([A-Z]\d{2}(?:\.\d{1,2})?)\b', raw)[:10]}


def cham_1_ca(llm_output_raw, gold3, gold_full, valid_3char):
    raw_list = extract_json_safe(llm_output_raw).get('top10_icd', [])
    top_10   = [c for c in (chuan_hoa_ma_icd(x) for x in raw_list) if c][:10]
    top_10_3 = [c[:3] for c in top_10]

    g3 = str(gold3).strip().upper()
    gf = str(gold_full).strip().upper()
    try:
        rank_3char = top_10_3.index(g3) + 1
    except ValueError:
        rank_3char = 0

    return {
        'top_10'              : ' || '.join(top_10),
        'top_10_3char'        : ' || '.join(top_10_3),
        'hit1'                : bool(top_10_3 and top_10_3[0] == g3),
        'hit5'                : g3 in top_10_3[:5],
        'hit10'               : g3 in top_10_3[:10],
        'hit1_full_icd'       : bool(top_10 and top_10[0] == gf),
        'rank_3char'          : rank_3char,
        'n_ma_parse_duoc'     : len(top_10),
        'n_valid_in_catalogue': sum(1 for c in top_10_3 if c in valid_3char),
    }


_r = cham_1_ca('{"top10_icd": ["K21.9","R10","XX","J32.","A09"]}',
               'R10', 'R10.4', valid_codes_3char)
assert _r['rank_3char'] == 2 and _r['n_ma_parse_duoc'] == 4, _r
print("Bo cham OK:", _r)

## 7. Nạp model — y nguyên cấu hình TN3

4-bit NF4, mirror `NousResearch`, `pad_token`, `TERMINATORS`. Ba thứ này đã bàn kỹ ở TN3;
đổi bất kỳ cái nào là V1 không còn tái lập được.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, os

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip() or None
except Exception:
    HF_TOKEN = (os.environ.get('HF_TOKEN') or '').strip() or None
print("Token HF:", "co" if HF_TOKEN else "khong (mirror NousResearch khong can token)")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Dang tai model mirror: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=HF_TOKEN,
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TERMINATORS = [tokenizer.eos_token_id]
for t in ("<|eot_id|>", "<|im_end|>", "<|end|>", "<|endoftext|>"):
    i = tokenizer.convert_tokens_to_ids(t)
    if isinstance(i, int) and i >= 0 and i != tokenizer.unk_token_id and i not in TERMINATORS:
        TERMINATORS.append(i)

print("Nap model xong! TERMINATORS =", TERMINATORS)

## 8. Hàm sinh theo variant

In [ ]:
def sinh_predictions_variant(query_text, variant):
    prompt = PROMPTS[variant].format(query_text=query_text)

    if tokenizer.chat_template:
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(text, return_tensors="pt",
                           add_special_tokens=False).to(model.device)
    else:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS[variant],
            do_sample=False,
            eos_token_id=TERMINATORS,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()


_thu = sinh_predictions_variant(df_test.iloc[0]['query_text'], 'V1_baseline')
print("Thu V1:", _thu[:200])
assert _thu, "Model tra ve rong."

## 9. Dry run 5 ca × 3 variant

Ba điều phải thấy trước khi chạy full:

1. **V1 khớp TN3 từng ca.** Nếu có `tn3_perquery_llama3_8b.csv` trong gói, cell dưới so
   `top_10` của 5 ca đầu với tệp TN3. Lệch dù chỉ một ca là cấu hình đã trôi — dừng, báo thầy.
2. **V3 không bị cắt.** JSON phải xuất hiện trong output, không đứt giữa chừng.
3. **V2 vẫn hiểu tiếng Việt.** Đọc mắt 5 ca xem mã top-1 có gần đúng nghĩa không.

In [ ]:
import time

tn3_ref = (pd.read_csv(f'{DATA}/tn3_perquery_llama3_8b.csv').set_index('case_id')
           if CO_TN3_PERQUERY else None)

for variant in ['V1_baseline', 'V2_english', 'V3_cot']:
    print(f"\n{'='*70}\n--- DRY RUN: {variant} (max_new_tokens={MAX_TOKENS[variant]}) ---")
    t0 = time.time()
    for _, row in df_test.head(5).iterrows():
        raw = sinh_predictions_variant(row['query_text'], variant)
        sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
        ghi = ''
        if variant == 'V1_baseline' and tn3_ref is not None and row['case_id'] in tn3_ref.index:
            khop = (sc['top_10'] == str(tn3_ref.loc[row['case_id'], 'top_10']))
            ghi  = '  [V1 vs TN3: KHOP]' if khop else '  [V1 vs TN3: *** LECH ***]'
        print(f"  {row['case_id']}: hit1={sc['hit1']}, "
              f"valid={sc['n_valid_in_catalogue']}/{sc['n_ma_parse_duoc']}{ghi}")
        print(f"    top10 : {sc['top_10_3char']}")
        print(f"    raw   : {raw[:120].replace(chr(10), ' / ')}")
    print(f"  Toc do: {(time.time()-t0)/5:.1f} giay/ca "
          f"-> uoc {(time.time()-t0)/5*181/60:.0f} phut cho 181 ca")

## 10. Chạy full — 3 variant × 181 ca

Lưu CSV ngay sau mỗi variant. Colab ngắt giữa chừng thì chạy lại cell này, các variant
đã có tệp sẽ được nạp lại từ đĩa thay vì chạy lại (`BO_QUA_NEU_CO_TEP = True`).

In [ ]:
from tqdm.auto import tqdm
import hashlib

BO_QUA_NEU_CO_TEP = True
all_variant_results = {}

for variant in ['V1_baseline', 'V2_english', 'V3_cot']:
    fname = f'{OUT}/tn5_perquery_llama3_{variant}.csv'

    if BO_QUA_NEU_CO_TEP and os.path.exists(fname):
        df_variant = pd.read_csv(fname)
        elapsed = float('nan')
        print(f"\n=== {variant}: da co tep, nap lai tu dia ===")
    else:
        print(f"\n=== RUNNING {variant} ===")
        t0, results = time.time(), []
        for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc=variant):
            raw = sinh_predictions_variant(row['query_text'], variant)
            sc  = cham_1_ca(raw, row['gold3'], row['gold'], valid_codes_3char)
            results.append({
                'case_id'       : row['case_id'],
                'source_channel': row['source_channel'],
                'gold3'         : str(row['gold3']).strip().upper(),
                'gold_full'     : str(row['gold']).strip().upper(),
                'voi_toi_duoc'  : row['voi_toi_duoc'],
                'variant'       : variant,
                'llm_raw'       : raw,
                **sc,
            })
        df_variant = pd.DataFrame(results)
        df_variant.to_csv(fname, index=False, encoding='utf-8-sig')
        elapsed = (time.time() - t0) / 60

    with open(fname, 'rb') as f:
        sha = hashlib.sha256(f.read()).hexdigest()

    hit1 = int(df_variant['hit1'].astype(bool).sum())
    print(f"  Hit@1 : {hit1}/181 = {100*hit1/181:.2f}%")
    print(f"  0 ma  : {(df_variant['n_ma_parse_duoc'] == 0).sum()} ca parse duoc 0 ma")
    print(f"  Time  : {elapsed:.1f} phut" if elapsed == elapsed else "  Time  : (nap lai)")
    print(f"  SHA   : {sha}")

    all_variant_results[variant] = {'df': df_variant, 'sha': sha, 'time_min': elapsed}

## 11. Kiểm tái lập V1 — hai mức

**Mức con số:** V1 Hit@1 có nằm trong ±0,6 pp quanh 12,71% không (ngưỡng mục 5.3 phiếu).

**Mức từng ca:** mạnh hơn nhiều. Greedy decoding là tất định, nên V1 phải trùng TN3
**từng ca một**, không chỉ trùng con số tổng. Hai lượt chạy khác nhau vẫn có thể tình cờ
ra cùng 23/181 mà sai ở mười ca khác nhau — mức con số không bắt được chuyện đó.

Sai khác nhỏ ở mức ca (1–3 ca) là chuyện bình thường do phi tất định của phép nhân
ma trận trên GPU và do phiên bản `bitsandbytes` khác nhau. Sai khác lớn thì cấu hình đã trôi.

In [ ]:
df_v1 = all_variant_results['V1_baseline']['df'].set_index('case_id')
v1_hit1_pct = 100 * df_v1['hit1'].astype(bool).sum() / 181
TN3_HIT1 = 12.71

print(f"V1 Hit@1 : {v1_hit1_pct:.2f}%   |   TN3 baseline: {TN3_HIT1}%   "
      f"|   lech {abs(v1_hit1_pct - TN3_HIT1):.2f} pp")
print("  => KHOP (nguong +-0.6 pp)" if abs(v1_hit1_pct - TN3_HIT1) < 0.6 else
      "  => LECH. Bao thay: co non-determinism hoac config drift (muc 6.1 phieu).")

V1_KHOP_TUNG_CA = None
if CO_TN3_PERQUERY:
    chung = df_v1.index.intersection(tn3_ref.index)
    khop_top10 = sum(1 for i in chung
                     if str(df_v1.loc[i, 'top_10']) == str(tn3_ref.loc[i, 'top_10']))
    khop_hit1  = sum(1 for i in chung
                     if bool(df_v1.loc[i, 'hit1']) == bool(tn3_ref.loc[i, 'hit1']))
    V1_KHOP_TUNG_CA = khop_top10
    print(f"\nMuc tung ca (n={len(chung)}):")
    print(f"  Trung top-10 nguyen chuoi : {khop_top10}/{len(chung)} "
          f"({100*khop_top10/len(chung):.1f}%)")
    print(f"  Trung nhan hit1           : {khop_hit1}/{len(chung)} "
          f"({100*khop_hit1/len(chung):.1f}%)")
    lech = [i for i in chung
            if str(df_v1.loc[i, 'top_10']) != str(tn3_ref.loc[i, 'top_10'])]
    if lech:
        print(f"\n  {len(lech)} ca lech, ba ca dau:")
        for i in lech[:3]:
            print(f"    {i}\n      TN3 : {str(tn3_ref.loc[i,'top_10'])[:90]}"
                  f"\n      V1  : {str(df_v1.loc[i,'top_10'])[:90]}")
else:
    print("\n(Khong co tn3_perquery_llama3_8b.csv trong goi -> bo qua kiem tung ca.)")

## 12. Thống kê chéo ba variant + McNemar + Holm

Mỗi variant so với MedKG-HRR of-record (`rank3 == 1`) trên **cả n=181 và n=118 reachable**.

**Về Holm:** phiếu nêu hai ngưỡng khác nhau — mục 5.3 viết `< 0,00625` (tức 0,05/8, so với
p **thô**), mục 6.1 viết `p > 0,0125` (so với p **đã nhân**). Bảng dưới in cả `mcnemar_p`
thô lẫn `holm_adj_m8 = p × 8` để chọn ngưỡng nào cũng đọc được, và thêm cột
`holm_tuan_tu` là Holm step-down đúng nghĩa trên ba variant.

In [ ]:
from scipy.stats import binomtest
from scipy import stats

def wilson_ci(k, n, alpha=0.05):
    if n == 0:
        return (0.0, 0.0)
    z = stats.norm.ppf(1 - alpha / 2)
    p = k / n
    den = 1 + z**2 / n
    c = (p + z**2 / (2*n)) / den
    h = z * ((p*(1-p)/n + z**2/(4*n**2))**0.5) / den
    return (max(0, (c-h)*100), min(100, (c+h)*100))

med       = pd.read_csv(f'{DATA}/independent_scored_perquery.csv').set_index('case_id')
medkg_hit = (med['rank3'] == 1).astype(int)
mask_reach = (med['voi_toi_duoc'] == True).reindex(med.index).fillna(False)

def mcnemar(u_hit, o_hit, mask=None):
    u = u_hit.reindex(med.index).fillna(0).astype(int)
    o = o_hit.reindex(med.index).fillna(0).astype(int)
    if mask is not None:
        u, o = u[mask], o[mask]
    b = int(((u == 1) & (o == 0)).sum())
    c = int(((u == 0) & (o == 1)).sum())
    p = binomtest(min(b, c), n=b+c, p=0.5).pvalue if (b+c) > 0 else 1.0
    return b, c, float(p)

variant_stats = []
for variant in ['V1_baseline', 'V2_english', 'V3_cot']:
    data = all_variant_results[variant]
    df_v = data['df'].set_index('case_id')
    v_hit = df_v['hit1'].astype(bool).astype(int)
    v_al  = v_hit.reindex(med.index).fillna(0).astype(int)

    b181, c181, p181 = mcnemar(v_hit, medkg_hit)
    b118, c118, p118 = mcnemar(v_hit, medkg_hit, mask=mask_reach)

    k    = int(v_al.sum())
    lo, hi = wilson_ci(k, 181)
    mrr  = df_v['rank_3char'].apply(lambda x: 1/x if x > 0 else 0).mean()

    variant_stats.append({
        'variant'           : variant,
        'max_new_tokens'    : MAX_TOKENS[variant],
        'hit1_pct_full'     : round(100 * k / 181, 2),
        'hit1_ci'           : f'[{lo:.2f}, {hi:.2f}]',
        'hit1_pct_reach'    : round(100 * v_al[mask_reach].sum() / 118, 2),
        'hit1_pct_outreach' : round(100 * v_al[~mask_reach].sum() / 63, 2),
        'hit5_pct'          : round(100 * df_v['hit5'].astype(bool).sum() / 181, 2),
        'mrr'               : round(float(mrr), 4),
        'n_parse_0_ma'      : int((df_v['n_ma_parse_duoc'] == 0).sum()),
        'avg_valid_top10'   : round(float(df_v['n_valid_in_catalogue'].mean()), 2),
        'mcnemar_n181_b'    : b181, 'mcnemar_n181_c': c181,
        'mcnemar_n181_p'    : round(p181, 4),
        'mcnemar_n118_b'    : b118, 'mcnemar_n118_c': c118,
        'mcnemar_n118_p'    : round(p118, 4),
        'holm_adj_m8'       : round(min(1.0, p118 * 8), 4),
        'sha256'            : data['sha'],
        'time_min'          : (round(data['time_min'], 1)
                               if data['time_min'] == data['time_min'] else None),
    })

# Holm step-down dung nghia tren 3 variant (n=118)
thu_tu = sorted(range(3), key=lambda i: variant_stats[i]['mcnemar_n118_p'])
truoc  = 0.0
for hang, i in enumerate(thu_tu):
    adj = min(1.0, max(truoc, variant_stats[i]['mcnemar_n118_p'] * (3 - hang)))
    truoc = adj
    variant_stats[i]['holm_tuan_tu'] = round(adj, 4)

df_summary = pd.DataFrame(variant_stats)
df_summary.to_csv(f'{OUT}/tn5_summary_variants.csv', index=False, encoding='utf-8-sig')

cot_in = ['variant', 'hit1_pct_full', 'hit1_ci', 'hit1_pct_reach', 'hit1_pct_outreach',
          'mrr', 'mcnemar_n118_b', 'mcnemar_n118_c', 'mcnemar_n118_p',
          'holm_adj_m8', 'holm_tuan_tu']
print("=== TN5 CROSS-VARIANT SUMMARY ===")
print(df_summary[cot_in].to_string(index=False))
print("\nMoc TN3 (V1 phai trung): Hit@1 12.71 | reach 17.80 | outreach 3.17 | "
      "n=118 b=20 c=5 p=0.0041")

## 13. Phương sai giữa ba variant và phân loại kịch bản

In [ ]:
hit1s = [v['hit1_pct_full'] for v in variant_stats]
spread = max(hit1s) - min(hit1s)

KICH_BAN_TN5 = ('A (<=2pp) — Finding C robust across prompts' if spread <= 2 else
                'B (2-5pp) — prompt-dependent, cung huong'    if spread <= 5 else
                'C (>5pp)  — prompt-fragile, PHAI reframe')

print("=" * 72)
print(f"Hit@1 ba variant : {min(hit1s):.2f}% - {max(hit1s):.2f}%   (spread {spread:.2f} pp)")
for v in variant_stats:
    print(f"   {v['variant']:<12} {v['hit1_pct_full']:>6.2f}%   "
          f"n=118 p={v['mcnemar_n118_p']:.4f}  holm(x8)={v['holm_adj_m8']:.4f}")
print(f"\nKICH BAN {KICH_BAN_TN5}")

mat_sig = [v['variant'] for v in variant_stats if v['mcnemar_n118_p'] >= 0.05]
mat_holm = [v['variant'] for v in variant_stats if v['mcnemar_n118_p'] >= 0.00625]
print(f"\nVariant mat significance tho (p >= 0.05)      : {mat_sig or 'khong co'}")
print(f"Variant khong qua Holm (p tho >= 0.05/8)      : {mat_holm or 'khong co'}")

# TN7 rieng: CoT co giup khong
d_cot = (df_summary.set_index('variant').loc['V3_cot', 'hit1_pct_full']
         - df_summary.set_index('variant').loc['V1_baseline', 'hit1_pct_full'])
print(f"\n=== TN7 (CoT ablation) ===")
print(f"V3_cot - V1_baseline = {d_cot:+.2f} pp")
print("  => CoT giup dang ke, bao thay (muc 6.1 phieu)." if d_cot > 5 else
      "  => CoT khong doi huong ket luan." if abs(d_cot) <= 5 else
      "  => CoT lam giam manh, dang ghi vao bai.")
print("=" * 72)

## 14. Bảng dán vào bài + ba ví dụ định tính

Mục 5.3.D phiếu: ba ca show output của cả ba variant cạnh nhau.

In [ ]:
dfs = {v: all_variant_results[v]['df'].set_index('case_id')
       for v in ['V1_baseline', 'V2_english', 'V3_cot']}
h = {v: dfs[v]['hit1'].astype(bool) for v in dfs}

ca_v1_hit_v3_miss = [i for i in med.index if h['V1_baseline'].get(i) and not h['V3_cot'].get(i)]
ca_v1_miss_khac_hit = [i for i in med.index if (not h['V1_baseline'].get(i))
                       and (h['V2_english'].get(i) or h['V3_cot'].get(i))]
ca_ca_ba_hit = [i for i in med.index if all(h[v].get(i) for v in dfs)]

def in_ca(cid):
    print(f"\n  {cid} | gold3={med.loc[cid,'gold3']} ({med.loc[cid,'gold']}) | "
          f"out-of-reach={not med.loc[cid,'voi_toi_duoc']}")
    print(f"    Hoi : {str(med.loc[cid,'query_text'])[:150]}")
    for v in dfs:
        print(f"    {v:<12}: hit1={bool(h[v].get(cid))} | "
              f"{str(dfs[v].loc[cid,'top_10_3char'])[:80]}")

for nhan, ds in [('(1) V1 dung, V3(CoT) sai — CoT khong giup', ca_v1_hit_v3_miss),
                 ('(2) V1 sai, V2 hoac V3 dung — prompt variant giup', ca_v1_miss_khac_hit),
                 ('(3) Ca ba variant cung dung — ca de', ca_ca_ba_hit)]:
    print(f"\n{'='*72}\n{nhan}   (co {len(ds)} ca)")
    if ds:
        in_ca(ds[0])
    else:
        print("  Khong co ca nao thuoc nhom nay.")

VI_DU = {'v1_hit_v3_miss': ca_v1_hit_v3_miss,
         'v1_miss_khac_hit': ca_v1_miss_khac_hit,
         'ca_ba_hit': ca_ca_ba_hit}

## 15. Ghi chú diễn giải — sinh sẵn `.docx` bốn mục A–D

Điền sẵn phần số. **Đọc lại và viết nhận định trước khi gửi.**

In [ ]:
from docx import Document

s = df_summary.set_index('variant')

doc = Document()
doc.add_heading('TN5 + TN7 — Ghi chú diễn giải: prompt sensitivity và CoT ablation', 0)
doc.add_paragraph(f"Model: {MODEL_ID} | 4-bit NF4 | greedy | n = 181 ca độc lập | "
                  f"catalogue 13.081 mã ICD-10 | SHA-256 prompt V1: {SHA_V1[:16]}…")

doc.add_heading('A. Kết quả tổng quan', level=1)
t = doc.add_table(rows=1, cols=6); t.style = 'Light Grid Accent 1'
for i, tieu_de in enumerate(['Variant', 'Hit@1 (%)', 'KTC 95%', 'Reachable (%)',
                             'Out-of-reach (%)', 'McNemar p (n=118)']):
    t.rows[0].cells[i].text = tieu_de
for v in ['V1_baseline', 'V2_english', 'V3_cot']:
    r = t.add_row().cells
    r[0].text = v
    r[1].text = f"{s.loc[v,'hit1_pct_full']}"
    r[2].text = f"{s.loc[v,'hit1_ci']}"
    r[3].text = f"{s.loc[v,'hit1_pct_reach']}"
    r[4].text = f"{s.loc[v,'hit1_pct_outreach']}"
    r[5].text = f"{s.loc[v,'mcnemar_n118_p']}"
doc.add_paragraph(f"Phương sai Hit@1: max − min = {spread:.2f} pp "
                  f"({min(hit1s):.2f}% – {max(hit1s):.2f}%).")
doc.add_paragraph(f"Kịch bản: {KICH_BAN_TN5}")

doc.add_heading('B. Kiểm tái lập V1', level=1)
doc.add_paragraph(
    f"V1 Hit@1 = {v1_hit1_pct:.2f}% so với TN3 baseline 12,71% — lệch "
    f"{abs(v1_hit1_pct - TN3_HIT1):.2f} pp "
    f"({'trong' if abs(v1_hit1_pct-TN3_HIT1) < 0.6 else 'NGOÀI'} ngưỡng ±0,6 pp).")
if V1_KHOP_TUNG_CA is not None:
    doc.add_paragraph(
        f"Mức từng ca: {V1_KHOP_TUNG_CA}/181 ca trùng nguyên chuỗi top-10 với "
        f"tn3_perquery_llama3_8b.csv ({100*V1_KHOP_TUNG_CA/181:.1f}%). "
        f"Greedy decoding là tất định nên đây là kiểm chặt hơn mức con số tổng.")
else:
    doc.add_paragraph("Không có tệp per-query TN3 trong gói nên chỉ kiểm được ở mức con số tổng.")

doc.add_heading('C. Độ bền của significance', level=1)
doc.add_paragraph(
    f"V1 vs MedKG-HRR (n=118): b={int(s.loc['V1_baseline','mcnemar_n118_b'])}, "
    f"c={int(s.loc['V1_baseline','mcnemar_n118_c'])}, "
    f"p={s.loc['V1_baseline','mcnemar_n118_p']} — mốc TN3 là b=20, c=5, p=0,0041.")
for v in ['V2_english', 'V3_cot']:
    doc.add_paragraph(
        f"{v}: b={int(s.loc[v,'mcnemar_n118_b'])}, c={int(s.loc[v,'mcnemar_n118_c'])}, "
        f"p={s.loc[v,'mcnemar_n118_p']}, Holm×8 = {s.loc[v,'holm_adj_m8']}, "
        f"Holm tuần tự = {s.loc[v,'holm_tuan_tu']}. "
        f"{'Còn significance' if s.loc[v,'mcnemar_n118_p'] < 0.05 else 'MẤT significance'} ở mức thô; "
        f"{'qua' if s.loc[v,'mcnemar_n118_p'] < 0.00625 else 'KHÔNG qua'} ngưỡng Holm 0,05/8.")
doc.add_paragraph(f"TN7 — CoT ablation: V3 − V1 = {d_cot:+.2f} pp.")

doc.add_heading('D. Ví dụ định tính', level=1)
for nhan, ds in [('(1) V1 đúng, V3 (CoT) sai', VI_DU['v1_hit_v3_miss']),
                 ('(2) V1 sai, V2/V3 đúng', VI_DU['v1_miss_khac_hit']),
                 ('(3) Cả ba variant đúng', VI_DU['ca_ba_hit'])]:
    doc.add_heading(f"{nhan} — {len(ds)} ca", level=2)
    if not ds:
        doc.add_paragraph('Không có ca nào thuộc nhóm này.'); continue
    cid = ds[0]
    doc.add_paragraph(f"{cid} | gold3 = {med.loc[cid,'gold3']} ({med.loc[cid,'gold']})")
    doc.add_paragraph(f"Hỏi: {str(med.loc[cid,'query_text'])[:300]}")
    for v in dfs:
        doc.add_paragraph(f"{v}: hit1={bool(h[v].get(cid))} | "
                          f"{str(dfs[v].loc[cid,'top_10_3char'])[:120]}")

doc.add_heading('E. Nhận định (người viết bổ sung)', level=1)
doc.add_paragraph('[…]')

doc.save(f'{OUT}/tn5_ghichu_diengiai.docx')
print('Da sinh:', f'{OUT}/tn5_ghichu_diengiai.docx')

## 16. Băm SHA-256 và đóng gói — 6 tệp theo mục 5.1

In [ ]:
import shutil

BAT_BUOC = ['tn5_perquery_llama3_V1_baseline.csv',
            'tn5_perquery_llama3_V2_english.csv',
            'tn5_perquery_llama3_V3_cot.csv',
            'tn5_summary_variants.csv',
            'tn5_ghichu_diengiai.docx']

lines = []
for f in sorted(os.listdir(OUT)):
    if f.endswith('SHA256.txt'):
        continue
    with open(f'{OUT}/{f}', 'rb') as fh:
        lines.append(f'{hashlib.sha256(fh.read()).hexdigest()}  {f}')
with open(f'{OUT}/tn5_SHA256.txt', 'w', encoding='utf-8', newline='\n') as fh:
    fh.write('\n'.join(lines) + '\n')
print('\n'.join(lines))

thieu = [f for f in BAT_BUOC if not os.path.exists(f'{OUT}/{f}')]
print('\nDU 6 TEP BAT BUOC (tn5_SHA256.txt la tep thu 6).' if not thieu
      else f'\nTHIEU: {thieu}')

shutil.make_archive('/content/TN5_TN7_KetQua', 'zip', OUT)
from google.colab import files
files.download('/content/TN5_TN7_KetQua.zip')

## 17. Checklist trước khi gửi

**Báo ngay thầy** (mục 6.1 phiếu) nếu:

| Điều kiện | Lý do |
|---|---|
| V1 Hit@1 lệch TN3 > 1 pp | Non-determinism hoặc config drift — ảnh hưởng cả TN3 |
| Spread Hit@1 > 5 pp | Kịch bản C — Finding C phải reframe |
| Variant nào mất significance | Central claim yếu |
| V3 (CoT) tăng > 5 pp so V1 | CoT hiệu quả — có thể đổi variant chính |
| Một variant chạy > 3 giờ | Quota Colab sắp hết |

**Sự cố hay gặp**

- **V3 bị cắt JSON > 20% số ca** → tăng `MAX_TOKENS['V3_cot']` lên 800; kiểm xem regex
  fallback trong `extract_json_safe` có vớt được mã từ phần văn xuôi không.
- **V2 sinh nhiều mã vô lệ hơn V1** → Llama-3 yếu ở cross-lingual medical; đọc mắt 5 ca
  xem top-1 có gần nghĩa không. Cột `avg_valid_top10` trong `tn5_summary_variants.csv`
  là chỗ nhìn nhanh.
- **Colab timeout** → chạy lại cell Phần 10, `BO_QUA_NEU_CO_TEP = True` sẽ giữ variant đã xong.